# Hands-on 27 (Bab 11): Ekspor Model Sederhana ke Format Ringan

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Dua jalur deployment edge yang realistis:
- **Jalur A:** Random Forest scikit-learn -> joblib (untuk Raspberry Pi dengan Python).
- **Jalur B:** MLP kecil Keras -> .keras -> TensorFlow Lite (untuk inferensi edge).

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalur A: Ekspor scikit-learn (joblib)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/export_sklearn_model.py"], capture_output=True, text=True)
print(r.stdout)

[INFO] Akurasi test model: 0.9784
[SUKSES] Model scikit-learn disimpan ke: outputs/models/room_status_rf.joblib
[INFO] Ukuran file: 435297 bytes



## 2. Jalur B: Keras -> TFLite

In [3]:
r = subprocess.run([sys.executable, "scripts/export_keras_to_tflite.py"], capture_output=True, text=True)
print(r.stdout)

Saved artifact at '/tmp/tmpbmgw_zg8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 17), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140583773064000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583773068928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583773073680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583773075792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583773074208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583773262192: TensorSpec(shape=(), dtype=tf.resource, name=None)
[SUKSES] Model Keras disimpan ke: outputs/models/room_binary.keras
[SUKSES] Model TFLite disimpan ke: outputs/models/room_binary.tflite
[INFO] Ukuran file TFLite: 3884 bytes



## 3. Verifikasi Inferensi dari File Model

In [4]:
import joblib
import numpy as np
import pandas as pd

# Inferensi ulang dari model joblib
model = joblib.load("outputs/models/room_status_rf.joblib")
X_test = pd.read_csv("data/processed/classification_status/X_test.csv")
print("[INFO] Prediksi joblib (5 baris):", list(model.predict(X_test.head())))

# Inferensi ulang dari model TFLite (batch dim default 1: resize dulu ke 5 baris)
import tensorflow as tf
interpreter = tf.lite.Interpreter(model_path="outputs/models/room_binary.tflite")
Xb = pd.read_csv("data/processed/classification_room/X_test.csv").values.astype("float32")[:5]
inp_detail = interpreter.get_input_details()[0]
interpreter.resize_tensor_input(inp_detail["index"], Xb.shape)
interpreter.allocate_tensors()
interpreter.set_tensor(inp_detail["index"], Xb)
interpreter.invoke()
out = interpreter.get_output_details()[0]["index"]
print("[INFO] Probabilitas TFLite (5 baris):", interpreter.get_tensor(out).flatten().round(3).tolist())

[INFO] Prediksi joblib (5 baris): ['normal', 'normal', 'normal', 'normal', 'normal']


I0000 00:00:1792631090.218889    5158 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792631090.251242    5158 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1792631091.122502    5158 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[INFO] Probabilitas TFLite (5 baris): [0.7379999756813049, 0.003000000026077032, 0.0, 0.0, 0.0]


/home/sandbox/iot-ml-book/.venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## Penutup

Training dan deployment adalah dua tahap berbeda. Model joblib (sekitar 425 KB) siap
untuk Raspberry Pi dengan Python biasa; model TFLite hanya sekitar 3.9 KB dan bisa
berjalan dengan interpreter ringan tanpa TensorFlow penuh. Keduanya terverifikasi bisa
dimuat ulang dan dipakai inferensi dari file hasil ekspor di `outputs/models/`.